# EcoSort — Clasificador de residuos (TrashNet + MobileNetV2) desde cero

Entrena un clasificador de 6 clases con **transfer learning sobre MobileNetV2** (ADR 0001: fine-tuning
sobre un modelo preentrenado) y lo exporta a **TFLite int8** para la Raspberry Pi 3.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
Después: *Entorno de ejecución → Ejecutar todas*.

Al final se descarga un `.zip` con:
- `ecosort_int8.tflite` → el que va a la Pi
- `ecosort_fp32.tflite` → referencia para comparar velocidad
- `labels.txt` → nombres de las clases en el orden del modelo
- `ecosort_mobilenetv2.keras` → modelo completo, para reentrenar más adelante

El preprocesado (escalar a [-1, 1]) está **dentro del modelo**: en la Pi se le pasa la imagen RGB
con valores 0–255 y listo.

## 1. Verificar GPU

In [ ]:
!nvidia-smi -L || echo "SIN GPU: activala en Entorno de ejecución -> Cambiar tipo de entorno"
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Descargar el dataset (desde el repo del grupo)

In [ ]:
!git clone --depth 1 -q https://github.com/tpII/2026-g5-ecosort.git
from pathlib import Path

# Busca la carpeta que contiene las subcarpetas de clases (cardboard, glass, ...)
candidatas = [p.parent for p in Path("2026-g5-ecosort").rglob("cardboard") if p.is_dir()]
assert candidatas, "No encontré el dataset en el repo"
DATA_DIR = candidatas[0]
print("Dataset:", DATA_DIR)

EXT = {".jpg", ".jpeg", ".png"}
CLASES = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
for c in CLASES:
    print(f"  {c:10s} {sum(1 for f in (DATA_DIR / c).iterdir() if f.suffix.lower() in EXT)} imágenes")

## 3. Separar train / validación / test (70/15/15, estratificado)

In [ ]:
import random
import numpy as np

IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

rng = random.Random(SEED)
splits = {"train": [], "val": [], "test": []}
for i, c in enumerate(CLASES):
    files = sorted(str(f) for f in (DATA_DIR / c).iterdir() if f.suffix.lower() in EXT)
    rng.shuffle(files)
    a, b = int(len(files) * 0.70), int(len(files) * 0.85)
    splits["train"] += [(f, i) for f in files[:a]]
    splits["val"]   += [(f, i) for f in files[a:b]]
    splits["test"]  += [(f, i) for f in files[b:]]
print({k: len(v) for k, v in splits.items()})

def cargar(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)          # float32 en 0..255 (igual que en la Pi)
    return img, label

aumentar = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.2),
])

def hacer_ds(items, entrenar=False, batch=True):
    paths, labels = zip(*items)
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    ds = ds.map(cargar, num_parallel_calls=AUTOTUNE).cache()
    if entrenar:
        ds = ds.shuffle(len(items), seed=SEED)   # después del cache: mezcla distinto cada época
    if not batch:
        return ds
    ds = ds.batch(BATCH)
    if entrenar:
        ds = ds.map(lambda x, y: (tf.clip_by_value(aumentar(x, training=True), 0, 255), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = hacer_ds(splits["train"], entrenar=True)
val_ds   = hacer_ds(splits["val"])
test_ds  = hacer_ds(splits["test"])

# Pesos por clase: "trash" tiene muchas menos imágenes que el resto
conteo = np.bincount([l for _, l in splits["train"]], minlength=len(CLASES))
class_weight = {i: len(splits["train"]) / (len(CLASES) * n) for i, n in enumerate(conteo)}
print("class_weight:", {CLASES[i]: round(w, 2) for i, w in class_weight.items()})

## 4. Modelo: MobileNetV2 preentrenado + cabeza nueva

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False,
                                         weights="imagenet")
base.trainable = False

entradas = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(entradas)   # 0..255 -> -1..1
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
salidas = tf.keras.layers.Dense(len(CLASES), activation="softmax")(x)
model = tf.keras.Model(entradas, salidas)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
parar = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                         restore_best_weights=True)
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weight, callbacks=[parar])

## 5. Fine-tuning: descongelar las últimas capas de MobileNetV2

In [ ]:
base.trainable = True
for capa in base.layers[:-40]:
    capa.trainable = False
for capa in base.layers:                      # BatchNorm siempre congelado
    if isinstance(capa, tf.keras.layers.BatchNormalization):
        capa.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=15,
                  class_weight=class_weight, callbacks=[parar])

## 6. Evaluar en test (imágenes que el modelo nunca vio)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

y_true = np.array([l for _, l in splits["test"]])
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
print(f"Accuracy test (Keras): {(y_true == y_pred).mean():.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASES, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(CLASES)), CLASES, rotation=45)
ax.set_yticks(range(len(CLASES)), CLASES)
for i in range(len(CLASES)):
    for j in range(len(CLASES)):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusión (test)")
plt.tight_layout(); plt.show()

## 7. Exportar a TFLite (fp32 e int8)

In [ ]:
model.save("ecosort_mobilenetv2.keras")
model.export("saved_model_ecosort")

conv = tf.lite.TFLiteConverter.from_saved_model("saved_model_ecosort")
Path("ecosort_fp32.tflite").write_bytes(conv.convert())

def dataset_representativo():
    for img, _ in hacer_ds(splits["train"], batch=False).take(200):
        yield [tf.expand_dims(img, 0)]

conv = tf.lite.TFLiteConverter.from_saved_model("saved_model_ecosort")
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = dataset_representativo   # entrada/salida quedan en float32
Path("ecosort_int8.tflite").write_bytes(conv.convert())

Path("labels.txt").write_text("\n".join(CLASES) + "\n")
for f in ["ecosort_mobilenetv2.keras", "ecosort_fp32.tflite", "ecosort_int8.tflite"]:
    print(f"{f:28s} {Path(f).stat().st_size / 1e6:.1f} MB")

## 8. Verificar que el TFLite int8 no perdió precisión

In [ ]:
def evaluar_tflite(ruta):
    interp = tf.lite.Interpreter(model_path=ruta)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    aciertos = 0
    for img, label in hacer_ds(splits["test"], batch=False):
        interp.set_tensor(inp["index"], tf.expand_dims(img, 0).numpy().astype(np.float32))
        interp.invoke()
        aciertos += int(interp.get_tensor(out["index"])[0].argmax() == int(label))
    return aciertos / len(splits["test"])

for f in ["ecosort_fp32.tflite", "ecosort_int8.tflite"]:
    print(f"{f:22s} accuracy test: {evaluar_tflite(f):.3f}")

## 9. Descargar los archivos

In [ ]:
!zip -q ecosort_modelo.zip ecosort_int8.tflite ecosort_fp32.tflite labels.txt ecosort_mobilenetv2.keras
from google.colab import files
files.download("ecosort_modelo.zip")